# Lab 11-07 · The Discrete-Time Fourier Transform

Companion to [chapter 11-07](../../chapters/11-07-discrete-time-fourier-transform.md). Four parts: the DTFT by summation and by FFT, measured frequency responses, windowed filter design, and resolution. Run the cells in order.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

## Part 1 — The DTFT by summation and by FFT

For a finite sequence, the FFT of the sequence zero-padded to $N$ points gives the DTFT at $\omega_k = 2\pi k/N$.

In [2]:
def dtft(x, w, n0=0):
    n = n0 + np.arange(len(x))
    return np.array([np.sum(x * np.exp(-1j * ww * n)) for ww in np.atleast_1d(w)])

x = np.array([1.0, 2.0, 1.0])
N = 64
Xfft = np.fft.fft(x, N)
wk = 2 * np.pi * np.arange(N) / N
print("largest difference, FFT vs direct sum:", np.max(np.abs(Xfft - dtft(x, wk))))
print("|X| at 0, pi/2, pi:", np.round(np.abs(dtft(x, [0, np.pi / 2, np.pi])), 6))

largest difference, FFT vs direct sum: 2.7822109252321576e-15
|X| at 0, pi/2, pi: [4. 2. 0.]


## Part 2 — Measuring a frequency response

Drive the five-point average at 500 SPS with sinusoids and compare the steady-state amplitude with $\lvert H(e^{j\omega})\rvert$.

In [3]:
fs = 500.0
h = np.ones(5) / 5
n = np.arange(2000)
for f in (50, 100, 150, 200):
    y = np.convolve(np.cos(2 * np.pi * f / fs * n), h)[100:2000]
    w = 2 * np.pi * f / fs
    Hm = abs(np.sin(5 * w / 2) / (5 * np.sin(w / 2)))
    print(f"{f:3d} Hz: measured amplitude {np.max(np.abs(y)):.4f}, |H| = {Hm:.4f}")

 50 Hz: measured amplitude 0.6472, |H| = 0.6472
100 Hz: measured amplitude 0.0000, |H| = 0.0000
150 Hz: measured amplitude 0.2472, |H| = 0.2472
200 Hz: measured amplitude 0.0000, |H| = 0.0000


## Part 3 — Windowed filter design (guided exercise)

**Predict first:** lengthening a rectangular-window low-pass design (cutoff π/4) from 21 to 81 taps: does the overshoot shrink? Does the transition? Then run the cell, which also compares the Hamming window.

In [4]:
def design(L, window):
    M = (L - 1) // 2
    m = np.arange(-M, M + 1)
    hh = np.where(m == 0, 0.25, np.sin(np.pi * m / 4) / (np.pi * np.where(m == 0, 1, m)))
    if window == "hamming":
        hh = hh * np.hamming(L)
    w = np.linspace(0, np.pi, 30001)
    H = np.cos(np.outer(w, m)) @ hh
    i9 = np.where(H >= 0.9)[0]
    i9 = i9[w[i9] <= np.pi / 4 + 0.3].max()
    i1 = np.argmax((w > np.pi / 4) & (H <= 0.1))
    return H[w <= np.pi / 4].max(), (w[i1] - w[i9]) / np.pi, 20 * np.log10(np.abs(H[w >= 0.5 * np.pi]).max())

for window in ("rectangular", "hamming"):
    for L in (21, 81, 201):
        pk, tw, sb = design(L, window)
        print(f"{window:11s} L = {L:3d}: peak {pk:.4f}, transition {tw:.4f} pi, stopband (w >= pi/2) {sb:6.1f} dB")

rectangular L =  21: peak 1.0727, transition 0.0811 pi, stopband (w >= pi/2)  -26.5 dB
rectangular L =  81: peak 1.0937, transition 0.0223 pi, stopband (w >= pi/2)  -42.0 dB
rectangular L = 201: peak 1.0911, transition 0.0089 pi, stopband (w >= pi/2)  -49.9 dB
hamming     L =  21: peak 1.0040, transition 0.1819 pi, stopband (w >= pi/2)  -49.8 dB
hamming     L =  81: peak 1.0017, transition 0.0456 pi, stopband (w >= pi/2)  -64.6 dB


hamming     L = 201: peak 1.0018, transition 0.0183 pi, stopband (w >= pi/2)  -72.0 dB


## Part 4 — Resolution and zero-padding

Two tones 50 Hz apart at 8 kHz. Evaluate the DTFT magnitude of records of different lengths on a fine grid, and compare its value midway between the tones (1025 Hz) with its value at the tones: a ratio near 1 means one merged peak; a ratio well below 1 means two resolved peaks. Zero-padding does not change the DTFT at all, only how finely an FFT samples it.

In [5]:
fs = 8000.0

def mag(N, freqs):
    m = np.arange(N)
    x = np.cos(2 * np.pi * 1000 * m / fs) + np.cos(2 * np.pi * 1050 * m / fs)
    return np.array([abs(np.sum(x * np.exp(-2j * np.pi * fq * m / fs))) for fq in freqs]) / N

for N in (64, 160, 512):
    at = mag(N, [1000, 1025, 1050])
    print(f"N = {N:3d} ({N / fs * 1e3:5.1f} ms): |X| at 1025 Hz / at the tones = {at[1] / max(at[0], at[2]):.3f}")
N = 64
padded = np.abs(np.fft.fft(np.cos(2 * np.pi * 1000 * np.arange(N) / fs) + np.cos(2 * np.pi * 1050 * np.arange(N) / fs), 16 * N)) / N
k = int(round(1025 / fs * 16 * N))
print(f"zero-padded 16x FFT at 1025 Hz: {padded[k]:.4f}; direct DTFT there: {mag(N, [1025])[0]:.4f}")

N =  64 (  8.0 ms): |X| at 1025 Hz / at the tones = 1.057
N = 160 ( 20.0 ms): |X| at 1025 Hz / at the tones = 0.028
N = 512 ( 64.0 ms): |X| at 1025 Hz / at the tones = 0.107
zero-padded 16x FFT at 1025 Hz: 0.7731; direct DTFT there: 0.7735


**What to take away.** The FFT samples the DTFT. A frequency response can be measured with sinusoids, exactly as the eigenfunction property predicts. Truncation ripple does not go away with length; windows trade it for transition width. And resolution needs observation time: zero-padding smooths the picture without separating close tones.